# Post-selection

`qarp.PostSelection` conditions results *after the fact* — it never touches circuits or engines.  It applies to readout distributions (sampled or `qarp.EXACT`) and to statevectors, and always reports the **success rate** (the probability mass that survived the condition).

Conventions: distribution keys are LSB-first tuples (qubit `q` at position `q`); statevector index bit `q` is qubit `q`.

In [ ]:
import numpy as np

from qarp import EXACT, PostSelection
from qarp.algorithms import Sampler
from qarp.blocks import SimpleBlock
from qarp.engines import QarpEngine

## Fixed bits on a readout distribution

Fixed-bit conditions collapse the selected qubits to a definite value, so those qubits are removed from the output keys (surviving qubits keep their ascending order).

In [ ]:
# A 7-qubit readout distribution.
readout = {
    (1, 0, 1, 0, 0, 0, 1): 0.55,
    (1, 1, 0, 1, 0, 0, 1): 0.30,
    (0, 0, 1, 0, 0, 0, 0): 0.15,
}

# Keep outcomes where qubits 3, 4, 5 read 0; those qubits then drop out.
ps = PostSelection({3: 0, 4: 0, 5: 0})
out = ps.apply(readout)

print("conditioned distribution:", {k: round(v, 4) for k, v in out.distribution.items()})
print("success rate:", out.success_rate)

## From a circuit

The same spec applies to any `Sampler` output — finite shots or the exact Born distribution.  On a Bell pair, conditioning qubit 0 on 0 forces qubit 1 to 0.

In [ ]:
bell = SimpleBlock(2, name="bell")
bell.h(0)
bell.cx(0, 1)
bell.build()

ps = PostSelection({0: 0})

for n_shots in (4000, EXACT):
    sampler = Sampler(ket=bell, n_shots=n_shots)
    engine = QarpEngine(seed=7)
    engine.build([sampler])
    out = ps.apply(engine.run()[0])
    print(f"n_shots={n_shots}:  distribution={out.distribution}  success={out.success_rate:.4f}")

## Statevectors

`apply_statevector` projects onto the condition and renormalises, returning the conditional state and the success probability $\lVert P|\psi\rangle\rVert^2$.  For fixed bits the selected qubits factor out, so the state compresses to the surviving register.

In [ ]:
ghz = SimpleBlock(3, name="ghz")
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)
ghz.build()

sv = ghz.statevector()

conditional, p = PostSelection({0: 0}).apply_statevector(sv, 3)
print("success probability:", p)
print("conditional state of qubits 1, 2:", np.round(conditional, 6))

## Sector post-selection

`hamming_weight` / `parity` select a *symmetry sector* (e.g. particle number under Jordan-Wigner) rather than fixed values.  A sector is a subspace — the selected qubits stay entangled inside it — so the output keeps the **full register width**.

In [ ]:
# A state with particle-number leakage: mostly N=2, some N=1 amplitude.
sv = np.zeros(16, dtype=complex)
sv[0b0011] = np.sqrt(0.45)  # qubits 0, 1 occupied
sv[0b0101] = np.sqrt(0.45)  # qubits 0, 2 occupied
sv[0b0001] = np.sqrt(0.10)  # N=1 leakage

ps_n2 = PostSelection.hamming_weight(range(4), k=2)

projected, p = ps_n2.apply_statevector(sv, 4)
print("N=2 sector weight:", round(p, 4))
print("projected state norm:", round(float(np.linalg.norm(projected)), 4))

born = {tuple(int(i >> q) & 1 for q in range(4)): abs(a) ** 2
        for i, a in enumerate(sv) if abs(a) > 0}
out = ps_n2.apply(born)
print("conditioned distribution:", {k: round(float(v), 4) for k, v in out.distribution.items()})

## Exploring success rates

Because the spec is reusable, it slots directly into parameter sweeps.  Here the post-selection succeeds with probability $\cos^2(\theta/2)$ by construction, and the `EXACT` readout reproduces it to machine precision — including the vanishing-support point at $\theta = \pi$, which reports rate 0 instead of raising.

In [ ]:
ps = PostSelection({1: 0})

print(" theta/pi   success   cos^2(theta/2)")
for theta in np.linspace(0, np.pi, 5):
    ket = SimpleBlock(2)
    ket.ry(0, theta)
    ket.cx(0, 1)
    ket.build()
    sampler = Sampler(ket=ket, n_shots=EXACT)
    engine = QarpEngine()
    engine.build([sampler])
    rate = ps.success_rate(engine.run()[0])
    print(f"{theta / np.pi:8.2f}   {rate:7.4f}   {np.cos(theta / 2) ** 2:12.4f}")